# Ingestão de dados — API HidroWebService (ANA)

Fase 1 do projeto: substituir o download manual de CSV por chamadas diretas
à API [HidroWebService](https://www.ana.gov.br/hidrowebservice/swagger-ui/index.html)
da ANA.

Este notebook é **exploração** (ver `README.md`): serve para descobrir o formato
real das respostas e desenhar a ingestão. O código estável e testado vai para
`data/src/`, seguindo o mesmo padrão do parser (`data/src/parse.py`).

**Atenção — a API da ANA é instável.** Durante os testes deste notebook, o
servidor respondeu `503 Service Unavailable` várias vezes sem motivo aparente.
Se uma célula falhar assim, espere alguns segundos e rode de novo — não é
necessariamente um erro seu.

**Atenção 2 — não reautentique em excesso.** O manual oficial avisa que
requisições de autenticação em alta frequência podem levar ao **bloqueio
automático do IP** pela ANA. O token dura 60 minutos: autentique uma vez por
sessão (célula 2) e reaproveite a variável `token` nas células seguintes.


In [1]:
import sys
from pathlib import Path
import json
import time
import pandas as pd
import requests


RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "data" / "src"))
BASE_URL = "https://www.ana.gov.br/hidrowebservice/EstacoesTelemetricas"

from ana_auth import autenticar, AutenticacaoError


/Users/boninixs/Documents/TCC/registro_ana_sc/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 1. Autenticação na API


In [2]:
def autenticar_com_retentativas(tentativas=3, espera_s=8):
    ultimo_erro = None
    for i in range(1, tentativas + 1):
        try:
            return autenticar()
        except (requests.exceptions.ReadTimeout, requests.exceptions.HTTPError) as e:
            ultimo_erro = e
            print(f"tentativa {i}/{tentativas} falhou ({e!r}); aguardando {espera_s}s...")
            time.sleep(espera_s)
    raise ultimo_erro


token = autenticar_com_retentativas()
headers = {"Authorization": f"Bearer {token}"}
print("Token obtido, válido por ~60 min.")


Token obtido, válido por ~60 min.


## 2. Inventário de estações — Santa Catarina

Rota `HidroInventarioEstacoes/v1`, filtrando por UF. Serve para descobrir quais
estações existem em SC e seus códigos (ex.: 2749005 = Ibirama, nossa estação
de referência — ver `docs/DECISOES.md`, Fase 0).


In [3]:
resp = requests.get(
    f"{BASE_URL}/HidroInventarioEstacoes/v1",
    headers=headers,
    params={"Unidade Federativa": "SC"},
    timeout=90,
)
resp.raise_for_status()
corpo = resp.json()
print("status:", corpo.get("status"), "| nº estações:", len(corpo.get("items") or []))

estacoes_sc = pd.json_normalize(corpo["items"])
estacoes_sc.head()


status: OK | nº estações: 1808


,Altitude,Area_Drenagem,Bacia_Nome,Codigo_Adicional,Codigo_Operadora_Unidade_UF,Data_Periodo_Climatologica_Fim,Data_Periodo_Climatologica_Inicio,Data_Periodo_Desc_Liquida_Fim,Data_Periodo_Desc_liquida_Inicio,Data_Periodo_Escala_Fim,...,Tipo_Rede_Curso_Dagua,Tipo_Rede_Energetica,Tipo_Rede_Estrategica,Tipo_Rede_Navegacao,Tipo_Rede_Qual_Agua,Tipo_Rede_Sedimentos,UF_Estacao,UF_Nome_Estacao,codigobacia,codigoestacao
0,3.0,None,"ATLÂNTICO,TRECHO LESTE",None,None,None,None,None,None,None,...,0,None,None,None,0,0,SC,SANTA CATARINA,5,2748094
1,None,None,RIO PARANÁ,None,None,None,None,None,None,None,...,0,1,0,0,0,0,SC,SANTA CATARINA,6,2045036
2,800.0,None,RIO PARANÁ,None,None,None,None,None,None,None,...,0,0,0,0,0,0,SC,SANTA CATARINA,6,2649011
3,862.0,None,RIO PARANÁ,574,None,1990-06-01 00:00:00.0,1990-06-01 00:00:00.0,None,None,None,...,0,0,0,0,0,0,SC,SANTA CATARINA,6,2649015
4,780.0,None,RIO PARANÁ,ANA,22,None,None,None,None,None,...,0,0,0,0,0,0,SC,SANTA CATARINA,6,2649016


In [4]:
cols_relevantes = [
    "codigoestacao", "Estacao_Nome", "Municipio_Nome", "Tipo_Estacao",
    "Operando", "Latitude", "Longitude",
    "Data_Periodo_Pluviometro_Inicio", "Data_Periodo_Pluviometro_Fim",
]
estacoes_sc[[c for c in cols_relevantes if c in estacoes_sc.columns]]


,codigoestacao,Estacao_Nome,Municipio_Nome,Tipo_Estacao,Operando,Latitude,Longitude,Data_Periodo_Pluviometro_Inicio,Data_Periodo_Pluviometro_Fim
0,2748094,ETE INSULAR,FLORIANÓPOLIS,Pluviometrica,1,-27.6006,-48.5561,2007-12-01 00:00:00.0,None
1,2045036,CGH ANIL JUSANTE,SANTANA DO JACARÉ,Pluviometrica,1,-20.8242,-45.0639,2015-12-01 00:00:00.0,None
2,2649011,BATEIAS DE BAIXO,CAMPO ALEGRE,Pluviometrica,0,-26.0667,-49.2667,1942-03-01 00:00:00.0,1967-05-01 00:00:00.0
3,2649015,RIO NEGRINHO,RIO NEGRINHO,Pluviometrica,1,-26.2478,-49.58,1990-06-01 00:00:00.0,None
4,2649016,RIO PRETO DO SUL,MAFRA,Pluviometrica,1,-26.2156,-49.6036,1951-01-01 00:00:00.0,None
...,...,...,...,...,...,...,...,...,...
1803,84955000,MARACANÃ,SOMBRIO,Fluviometrica,1,-29.0558,-49.6856,None,None
1804,84970000,PRAIA GRANDE,PRAIA GRANDE,Fluviometrica,1,-29.1953,-49.965,None,None
1805,84980000,POÇO NEGRO,SÃO JOÃO DO SUL,Fluviometrica,0,-29.2014,-49.9394,None,None
1806,84980010,POÇO NEGRO,SÃO JOÃO DO SUL,Fluviometrica,1,-29.2019,-49.9403,None,None


Depois de revisar aqui, salvamos o inventário em `data/interim/` — ele não é o
"CSV original da ANA" (isso é reservado a `data/raw/`), é um artefato derivado
da API que pode ser regenerado a qualquer momento.


In [5]:
caminho = RAIZ / "data" / "interim" / "estacoes_sc_inventario.parquet"
estacoes_sc.to_parquet(caminho, index=False)
print(f"salvo em {caminho}")


salvo em /Users/boninixs/Documents/TCC/registro_ana_sc/data/interim/estacoes_sc_inventario.parquet


In [6]:
df = pd.read_parquet("../data/interim/estacoes_sc_inventario.parquet")
df

,Altitude,Area_Drenagem,Bacia_Nome,Codigo_Adicional,Codigo_Operadora_Unidade_UF,Data_Periodo_Climatologica_Fim,Data_Periodo_Climatologica_Inicio,Data_Periodo_Desc_Liquida_Fim,Data_Periodo_Desc_liquida_Inicio,Data_Periodo_Escala_Fim,...,Tipo_Rede_Curso_Dagua,Tipo_Rede_Energetica,Tipo_Rede_Estrategica,Tipo_Rede_Navegacao,Tipo_Rede_Qual_Agua,Tipo_Rede_Sedimentos,UF_Estacao,UF_Nome_Estacao,codigobacia,codigoestacao
0,3.0,None,"ATLÂNTICO,TRECHO LESTE",None,None,None,None,None,None,None,...,0,None,None,None,0,0,SC,SANTA CATARINA,5,2748094
1,None,None,RIO PARANÁ,None,None,None,None,None,None,None,...,0,1,0,0,0,0,SC,SANTA CATARINA,6,2045036
2,800.0,None,RIO PARANÁ,None,None,None,None,None,None,None,...,0,0,0,0,0,0,SC,SANTA CATARINA,6,2649011
3,862.0,None,RIO PARANÁ,574,None,1990-06-01 00:00:00.0,1990-06-01 00:00:00.0,None,None,None,...,0,0,0,0,0,0,SC,SANTA CATARINA,6,2649015
4,780.0,None,RIO PARANÁ,ANA,22,None,None,None,None,None,...,0,0,0,0,0,0,SC,SANTA CATARINA,6,2649016
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1805,16.0,32.0,"ATLÂNTICO, TRECHO SUDESTE",EPAGRI,None,None,None,None,2005-04-01 00:00:00.0,None,...,4,0,0,0,3,0,SC,SANTA CATARINA,8,84955000
1806,None,339.0,"ATLÂNTICO, TRECHO SUDESTE",None,23,None,None,None,1986-04-01 00:00:00.0,None,...,3,0,0,0,3,0,SC,SANTA CATARINA,8,84970000
1807,None,378.0,"ATLÂNTICO, TRECHO SUDESTE",None,24,None,None,2018-12-01 00:00:00.0,2006-02-01 00:00:00.0,2018-12-01 00:00:00.0,...,3,0,0,0,3,0,SC,SANTA CATARINA,8,84980000
1808,None,None,"ATLÂNTICO, TRECHO SUDESTE",SCRS-7759-IE-16,23,None,None,None,None,None,...,3,0,0,0,3,0,SC,SANTA CATARINA,8,84980010


## 3. Ingestão dos CSVs brutos — estações pluviométricas (`data/raw/`)

Reaproveita a rota `HidroSerieChuva/v1` e a lógica de download em blocos de até
366 dias já implementada em `data/src/ana_ingest.py`, filtrando o inventário
para `Tipo_Estacao_Pluviometro == "1"` (876 das 1.808 estações de SC — as
fluviométricas ficam de fora nesta fase).

O CSV salvo em `data/raw/<codigoestacao>.csv` é o retorno **cru** da API, sem
parsing nem deduplicação — equivalente ao download manual do Portal Hidroweb,
que é o que `data/raw/` deve conter por convenção do projeto.

Teste aqui com poucas estações antes de rodar a ingestão completa: o processo
para as 876 estações é longo (uma chamada por janela de até 366 dias, por
estação) e mais adequado para rodar como script em background do que
célula a célula neste notebook.

In [7]:
from ana_ingest import carregar_inventario, SessaoAna, baixar_estacao, DIR_RAW, registrar_log

pluviometricas = carregar_inventario()
print(f"{len(pluviometricas)} estações pluviométricas no inventário de SC.")
pluviometricas.head()

877 estações pluviométricas no inventário de SC.


,codigoestacao,Estacao_Nome,ini,fim
1,2045036,CGH ANIL JUSANTE,2015-12-01,2026-09-09
874,2648002,LUIZ ALVES,1941-01-01,2026-09-09
884,2648013,SÃO FRANCISCO DO SUL,1923-01-01,2026-09-09
885,2648014,JOINVILLE (RVPSC),1938-05-01,2026-09-09
888,2648020,ITAPOCU,1976-03-01,2026-09-09


### 3.1 Teste com poucas estações

Baixa e salva de fato (mesma função usada na ingestão completa), mas só as 3
primeiras estações do inventário — serve para validar o formato do CSV cru
antes de rodar tudo.

In [8]:
DIR_RAW.mkdir(parents=True, exist_ok=True)
sessao = SessaoAna()

teste = pluviometricas.head(3)
for row in teste.itertuples(index=False):
    codigo, ini, fim = row.codigoestacao, row.ini.date(), row.fim.date()
    destino = DIR_RAW / f"{codigo}.csv"
    print(f"{codigo} ({row.Estacao_Nome}) — {ini}..{fim}")

    df_estacao = baixar_estacao(sessao, codigo, ini, fim, delay_s=0.4)
    if df_estacao.empty:
        registrar_log(codigo, "vazio", 0, "API não retornou registros")
        print("  vazio")
        continue

    df_estacao.to_csv(destino, index=False)
    registrar_log(codigo, "ok", len(df_estacao), "")
    print(f"  salvo: {destino.name} ({len(df_estacao)} registros)")

2045036 (CGH ANIL JUSANTE) — 2015-12-01..2026-09-09
  vazio
2648002 (LUIZ ALVES) — 1941-01-01..2026-09-09
  salvo: 2648002.csv (1692 registros)
2648013 (SÃO FRANCISCO DO SUL ) — 1923-01-01..2026-09-09
  salvo: 2648013.csv (320 registros)


In [ ]:
# conferir o formato salvo em data/raw/
raw_exemplo = DIR_RAW / f"{teste.iloc[0].codigoestacao}.csv"
pd.read_csv(raw_exemplo).head()

### 3.2 Ingestão completa (876 estações)

Depois de validar o formato acima, rode a ingestão completa **pelo terminal**,
não por célula — é um processo longo (horas) e o script já é idempotente
(pula estação cujo CSV já existe em `data/raw/`) e resumível caso caia no
meio:

```bash
python data/src/ana_ingest.py                # todas as 876 estações pluviométricas
python data/src/ana_ingest.py --delay 0.5     # pausa maior entre requisições
python data/src/ana_ingest.py --forcar        # rebaixar mesmo se o CSV já existir
```

Progresso e erros ficam em `data/interim/log_ingestao_raw.csv`.